In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [ ]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


*encode target variable*

In [ ]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})

In [ ]:
X = df["message"]
y = df["label"]

# Text preprocessing

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(X)

X_tfidf.shape

(5572, 8444)

# Baseline models

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
svm = SVC(kernel="linear", probability=True)

# Voting classifier


In [ ]:
from sklearn.ensemble import VotingClassifier

hard_voting = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    voting="hard"
)

soft_voting = VotingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    voting="soft"
)

# Stacking classifier

In [ ]:
from sklearn.ensemble import StackingClassifier

stacking = StackingClassifier(
    estimators=[("nb", nb), ("lr", lr), ("svm", svm)],
    final_estimator=LogisticRegression(max_iter=1000)
)

# AdaBoost with stumps

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),  # decision stump
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

# All models

In [ ]:
models = {
    "NaiveBayes": nb,
    "LogisticRegression": lr,
    "LinearSVM": svm,
    "HardVoting": hard_voting,
    "SoftVoting": soft_voting,
    "Stacking": stacking,
    "AdaBoost_Stumps": adaboost
}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

for model_name, model in models.items():
    precision, recall, f1, roc = [], [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english")),
            ("clf", model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)

        precision.append(precision_score(y_test, y_pred))
        recall.append(recall_score(y_test, y_pred))
        f1.append(f1_score(y_test, y_pred))

        if hasattr(pipeline.named_steps["clf"], "predict_proba"):
            y_prob = pipeline.predict_proba(X_test)[:, 1]
            roc.append(roc_auc_score(y_test, y_prob))

    results.append({
        "Model": model_name,
        "Precision": f"{np.mean(precision):.3f} ± {np.std(precision):.3f}",
        "Recall": f"{np.mean(recall):.3f} ± {np.std(recall):.3f}",
        "F1": f"{np.mean(f1):.3f} ± {np.std(f1):.3f}",
        "ROC-AUC": (
            f"{np.mean(roc):.3f} ± {np.std(roc):.3f}"
            if len(roc) > 0 else "N/A"
        )
    })

In [ ]:
comparison_df = pd.DataFrame(results)
comparison_df.to_csv("ensemble_comparison.csv", index=False)
comparison_df

,Model,Precision,Recall,F1,ROC-AUC
0,NaiveBayes,0.998 ± 0.003,0.782 ± 0.020,0.877 ± 0.013,0.988 ± 0.004
1,LogisticRegression,0.986 ± 0.009,0.726 ± 0.015,0.836 ± 0.011,0.991 ± 0.005
2,LinearSVM,0.978 ± 0.009,0.904 ± 0.020,0.939 ± 0.009,0.992 ± 0.004
3,HardVoting,0.987 ± 0.008,0.829 ± 0.006,0.901 ± 0.006,N/A
4,SoftVoting,0.985 ± 0.007,0.885 ± 0.023,0.932 ± 0.015,0.992 ± 0.004
5,Stacking,0.971 ± 0.008,0.929 ± 0.018,0.949 ± 0.010,0.992 ± 0.004
6,AdaBoost_Stumps,0.989 ± 0.013,0.245 ± 0.019,0.392 ± 0.024,0.897 ± 0.012


In [ ]:
final_model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", StackingClassifier(
        estimators=[
            ("nb", MultinomialNB()),
            ("lr", LogisticRegression(max_iter=1000)),
            ("svm", SVC(kernel="linear", probability=True))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        passthrough=True   # prevents all-zero predictions
    ))
])

In [ ]:
final_model.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer(stop_words='english')),
                ('clf',
                 StackingClassifier(estimators=[('nb', MultinomialNB()),
                                                ('lr',
                                                 LogisticRegression(max_iter=1000)),
                                                ('svm',
                                                 SVC(kernel='linear',
                                                     probability=True))],
                                    final_estimator=LogisticRegression(max_iter=1000),
                                    passthrough=True))])

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

confusion_matrix(y_test, y_pred)

array([[964,   2],
       [ 13, 136]])

In [ ]:
final_predictions = pd.DataFrame({
    "MessageId": np.arange(1, len(y_test) + 1),
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred,
    "Probability": y_prob
})

final_predictions.to_csv("final_model_predictions.csv", index=False)
final_predictions.head()

,MessageId,Actual,Predicted,Probability
0,1,0,0,0.006769
1,2,0,0,0.007704
2,3,0,0,0.006299
3,4,1,1,0.999018
4,5,0,0,0.006507


# Recommendation:
The stacking classifier is the best combining strategy as it achieved the highest F1-score and recall by learning how to optimally combine predictions from multiple base models. Soft voting performed well but treats all models equally, while AdaBoost with decision stumps was less effective for sparse TF-IDF text features.